# LTV Review Inspection

This notebook inspects the transferred LTV review files so you can see:

- what files are present
- what columns the LTV parquets contain
- what is stored in the review DB
- whether DB paths still point to the old machine
- whether the review queue should be empty or not
- whether saved app-state / queue filters may explain a blank UI


In [ ]:
from __future__ import annotations

import json
import sqlite3
from pathlib import Path
import sys

import pandas as pd

candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next(
    (p for p in candidate_roots if (p / "malca").is_dir() and (p / "malca" / "review" / "store.py").exists()),
    Path.cwd(),
)

for path in (REPO_ROOT, REPO_ROOT / "malca"):
    sp = str(path.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)

from malca.review.store import db_connect, query_queue

BASE_DIR_CANDIDATES = [
    REPO_ROOT / 'output' / 'ltv' / 'ltv',
    REPO_ROOT / 'output' / 'ltv',
    REPO_ROOT,
]
BASE_DIR = next(
    (
        p for p in BASE_DIR_CANDIDATES
        if (p / 'ltv_candidates.db').exists() or any(p.glob('*_pipeline.parquet'))
    ),
    BASE_DIR_CANDIDATES[0],
)
DB_PATH = BASE_DIR / 'ltv_candidates.db'
PIPELINE_FILES = sorted(BASE_DIR.glob('*_pipeline.parquet'))
CORE_FILES = sorted(p for p in BASE_DIR.glob('LTvar*.parquet') if not p.name.endswith('_pipeline.parquet'))
LIGHTCURVE_DIRS = [BASE_DIR / 'bundle_assets' / 'lightcurves', BASE_DIR / 'lightcurves']

print('BASE_DIR =', BASE_DIR)
print('DB_PATH  =', DB_PATH)
print('pipeline files =', len(PIPELINE_FILES))
print('core files     =', len(CORE_FILES))
print('lightcurve dirs =', LIGHTCURVE_DIRS)


## File Inventory

In [ ]:
inventory = []
for path in sorted(BASE_DIR.iterdir()):
    kind = 'dir' if path.is_dir() else 'file'
    size_mb = None if path.is_dir() else round(path.stat().st_size / 1024**2, 3)
    inventory.append({'name': path.name, 'kind': kind, 'size_mb': size_mb})

pd.DataFrame(inventory)

## Light-Curve Directory Checks

In [ ]:
rows = []
for d in LIGHTCURVE_DIRS:
    rows.append({
        'path': str(d),
        'exists': d.exists(),
        'is_dir': d.is_dir(),
        'n_files': sum(1 for p in d.iterdir() if p.is_file()) if d.exists() and d.is_dir() else 0,
    })
pd.DataFrame(rows)

## Heads Of LTV Pipeline Parquets

This gives you the file name, row count, key ID/path columns, and the first few rows.

In [ ]:
summary_rows = []
for path in PIPELINE_FILES:
    df = pd.read_parquet(path)
    summary_rows.append({
        'file': path.name,
        'rows': len(df),
        'has_ASAS-SN_ID': 'ASAS-SN ID' in df.columns,
        'has_asas_sn_id': 'asas_sn_id' in df.columns,
        'has_candidate_id': 'candidate_id' in df.columns,
        'has_lc_path': 'lc_path' in df.columns,
        'has_filter_reason': 'filter_reason' in df.columns,
    })
pd.DataFrame(summary_rows)

In [ ]:
for path in PIPELINE_FILES:
    print('=' * 100)
    print(path.name)
    df = pd.read_parquet(path)
    cols = [c for c in ['ASAS-SN ID', 'asas_sn_id', 'candidate_id', 'lc_path', 'Slope', 'max diff', 'filter_reason'] if c in df.columns]
    display(df[cols].head())

## Heads Of LTV Core Parquets

In [ ]:
for path in CORE_FILES:
    print('=' * 100)
    print(path.name)
    df = pd.read_parquet(path)
    cols = [c for c in ['ASAS-SN ID', 'asas_sn_id', 'candidate_id', 'lc_path', 'Slope', 'max diff'] if c in df.columns]
    display(df[cols].head())

## Review DB Schema And Counts

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
    display(tables)
    for table in tables['name']:
        count = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
        print(f'{table}: {count}')

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    schema = pd.read_sql_query("PRAGMA table_info(candidates)", conn)
schema

## Candidate Samples From The Review DB

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    sample = pd.read_sql_query(
        "SELECT candidate_id, asas_sn_id, lc_path, source_path, failed_any FROM candidates ORDER BY candidate_id LIMIT 10",
        conn,
    )
sample

In [ ]:
def resolve_local_name(path_text: str | None) -> str | None:
    if not path_text:
        return None
    return Path(str(path_text)).name

def find_local_copy(name: str | None) -> str | None:
    if not name:
        return None
    for d in LIGHTCURVE_DIRS:
        candidate = d / name
        if candidate.exists():
            return str(candidate)
    return None

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(
        "SELECT candidate_id, lc_path FROM candidates ORDER BY candidate_id LIMIT 20",
        conn,
    )

df['stored_name'] = df['lc_path'].map(resolve_local_name)
df['stored_path_exists_here'] = df['lc_path'].map(lambda x: Path(x).exists() if x else False)
df['local_bundle_match'] = df['stored_name'].map(find_local_copy)
df

## Compare DB Candidate IDs To IDs Derived From LTV Pipeline Files

This shows whether the DB and the pipeline outputs refer to the same objects.

In [ ]:
pipeline_candidate_ids = set()
for path in PIPELINE_FILES:
    df = pd.read_parquet(path, columns=['ASAS-SN ID'])
    pipeline_candidate_ids.update('ltv_' + df['ASAS-SN ID'].astype(str))

with sqlite3.connect(DB_PATH) as conn:
    db_candidate_ids = {
        row[0] for row in conn.execute('SELECT candidate_id FROM candidates')
    }

comparison = {
    'db_count': len(db_candidate_ids),
    'pipeline_count': len(pipeline_candidate_ids),
    'intersection': len(db_candidate_ids & pipeline_candidate_ids),
    'db_not_in_pipeline': len(db_candidate_ids - pipeline_candidate_ids),
    'pipeline_not_in_db': len(pipeline_candidate_ids - db_candidate_ids),
}
comparison

In [ ]:
missing_from_db = sorted(pipeline_candidate_ids - db_candidate_ids)[:20]
missing_from_pipeline = sorted(db_candidate_ids - pipeline_candidate_ids)[:20]
print('pipeline -> not in db (first 20):')
print(missing_from_db)
print()
print('db -> not in pipeline (first 20):')
print(missing_from_pipeline)

## Review Queue Diagnostics

This directly asks the same queue code the Dash app uses what should be visible.

In [ ]:
with db_connect(DB_PATH) as conn:
    queue_default = query_queue(conn, filters={}, ids_only=True)
    queue_unreviewed = query_queue(conn, filters={'only_unreviewed': True}, ids_only=True)
    queue_failed_false = query_queue(conn, filters={'require_failed_any_false': True}, ids_only=True)
    queue_both = query_queue(
        conn,
        filters={'only_unreviewed': True, 'require_failed_any_false': True},
        ids_only=True,
    )

pd.DataFrame([
    {'queue': 'default', 'size': len(queue_default)},
    {'queue': 'only_unreviewed', 'size': len(queue_unreviewed)},
    {'queue': 'require_failed_any_false', 'size': len(queue_failed_false)},
    {'queue': 'both', 'size': len(queue_both)},
])

In [ ]:
with db_connect(DB_PATH) as conn:
    queue_preview = query_queue(conn, filters={}, ids_only=False)
queue_preview.head(20)

## Persisted App State / Saved Filters

A blank queue in the UI can also happen if old saved filters are being restored from the DB.

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    app_state = pd.read_sql_query('SELECT key, value, updated_at FROM app_state ORDER BY updated_at DESC', conn)
app_state.head(50)

In [ ]:
def maybe_parse_json(value: str):
    try:
        return json.loads(value)
    except Exception:
        return value

parsed_state = app_state.copy()
if not parsed_state.empty:
    parsed_state['parsed_value'] = parsed_state['value'].map(maybe_parse_json)
parsed_state[['key', 'parsed_value', 'updated_at']].head(20)

## Quick Conclusions Checklist

Use the outputs above to answer these questions:

- Does the DB have candidates at all?
- Do the LTV parquets contain `candidate_id` or only `ASAS-SN ID`?
- Do DB `lc_path` values still point to the old machine?
- Is there a local copy of each light curve under `bundle_assets/lightcurves` or `lightcurves`?
- Does `query_queue(..., filters={})` return candidates?
- Are saved `app_state` filters scoping the UI down to zero rows?
